<p align="center"><img src="https://raw.githubusercontent.com/mbs57/universal-render/master/assets/universal_render.jpg" width="640"></p>

# universal-render — live demo

**Correct Hindi, Arabic, Bengali, Tamil, Thai, Chinese (and 16 more scripts) in Matplotlib — with zero font configuration.**

Matplotlib's own text engine cannot *shape* complex scripts: Devanagari and Bengali conjuncts break apart, Arabic letters don't join, Thai and Tamil vowels land in the wrong place — and often you just get tofu boxes (▯▯▯). `universal-render` routes text through Qt's HarfBuzz shaping engine and drops the correctly shaped result into your figures.

- GitHub: https://github.com/mbs57/universal-render
- PyPI: https://pypi.org/project/universal-render/

▶ **Just run every cell top to bottom** (Runtime → Run all). Works on Google Colab and Kaggle (enable *Internet* in Kaggle notebook settings). Takes about 2 minutes.

If you are here for the **user study**, run all cells, look at the figures — especially the ones in *your* language — and share your judgment via the links at the bottom. Thank you! 🙏

## 1. Setup — install the package and pan-Unicode fonts

In [ ]:
%pip -q install universal-render

# Noto fonts + Qt offscreen libraries (Linux / Colab / Kaggle)
import platform, subprocess
if platform.system() == 'Linux':
    subprocess.run('apt-get -qq update', shell=True, capture_output=True)
    subprocess.run(
        'apt-get -qq install -y fonts-noto-core fonts-noto-ui-core fonts-noto-cjk '
        'libegl1 libgl1 libxkbcommon0 libdbus-1-3',
        shell=True, capture_output=True)
    n = subprocess.run('fc-list | grep -ci noto', shell=True, capture_output=True, text=True)
    print('Noto font faces installed:', n.stdout.strip())
print('setup done')

## 2. Import and check the environment

In [ ]:
import os
os.environ.setdefault('QT_QPA_PLATFORM', 'offscreen')

import matplotlib.pyplot as plt
import numpy as np
import universal_render as ur

ur.init_renderer()
env = ur.check_environment()
print('universal-render', ur.__version__)
for key in ('colab', 'kaggle', 'headless', 'qt_qpa_platform'):
    print(f'  {key}: {env[key]}')

## 3. Self-test: which scripts render correctly on THIS machine?

Each of the 22 scripts gets a deliberately tricky sample word (conjuncts, vowel reordering, contextual joining). The test checks that a suitable font was found, that it actually has the glyphs, and that the render produced visible ink.

In [ ]:
results = ur.self_test()

## 4. Before / after — Matplotlib's engine vs universal-render

**Left column:** Matplotlib's own text engine (what you get today).
**Right column:** the same text through universal-render.

Matplotlib will print *"Glyph missing from font"* warnings below — that is the point being demonstrated.

In [ ]:
samples = {
    'Hindi':    'विश्वविद्यालय क्षेत्र',
    'Arabic':   'جامعة القاهرة',
    'Bengali':  'বিশ্ববিদ্যালয়ের শিক্ষার্থী',
    'Tamil':    'பல்கலைக்கழகம்',
    'Thai':     'มหาวิทยาลัย',
    'Telugu':   'విశ్వవిద్యాలయం',
    'Hebrew':   'אוניברסיטה',
    'Korean':   '대학교 연구',
    'English':  'University',
}
fig = ur.save_comparison_figure('comparison_before_after.png', samples=samples)
plt.show()

## 5. A real plot: six scripts, one figure, zero configuration

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(1, 7)
ax.plot(x, [3, 5, 4, 6, 5, 7], marker='o')

ur.set_multilingual_title(ax, 'बहुभाषी प्लॉट — متعدد اللغات', font_size=30)  # Hindi + Arabic
ur.set_multilingual_xlabel(ax, 'أشهر السنة', font_size=24)      # Arabic (RTL)
ur.set_multilingual_ylabel(ax, 'மாத விற்பனை', font_size=24)     # Tamil
ur.set_multilingual_xticks(
    ax, list(x),
    ['एक', 'দুই', 'மூன்று', 'أربعة', 'ห้า', 'six'],               # 6 scripts
    font_size=16,
)
ur.apply_multilingual_layout(fig, auto=True)
plt.show()

## 6. One-call annotated heatmap — native Devanagari digits, auto contrast

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
data = np.random.default_rng(3).random((3, 3))
ur.multilingual_heatmap(
    ax, data,
    value_format='.2f', value_script='Devanagari',              # native Hindi digits
    row_labels=['दिल्ली', 'مصر', 'กรุงเทพฯ'],                     # Hindi / Arabic / Thai
    col_labels=['গুণমান', 'வேகம்', 'Quality'],                   # Bengali / Tamil / English
    title='गुणवत्ता मैट्रिक्स',
    cell_font_size=30,
)
plt.show()

## 7. Language gallery — a fully localized plot for each language

Same code, six languages. Title, axis labels, month names, and even the numeric y-axis digits are localized — each string picks its own font automatically. **Find your language below.**

In [ ]:
GALLERY = {
    'Hindi': dict(script='Devanagari', title='मासिक बिक्री रिपोर्ट',
                  xlabel='महीना', ylabel='मूल्य',
                  months=['जनवरी', 'फ़रवरी', 'मार्च', 'अप्रैल']),
    'Arabic': dict(script='Arabic', title='تقرير المبيعات الشهرية',
                   xlabel='الشهر', ylabel='القيمة',
                   months=['يناير', 'فبراير', 'مارس', 'أبريل']),
    'Thai': dict(script='Thai', title='รายงานยอดขายรายเดือน',
                 xlabel='เดือน', ylabel='มูลค่า',
                 months=['มกราคม', 'กุมภาพันธ์', 'มีนาคม', 'เมษายน']),
    'Tamil': dict(script='Tamil', title='மாதாந்திர விற்பனை அறிக்கை',
                  xlabel='மாதம்', ylabel='மதிப்பு',
                  months=['ஜனவரி', 'பிப்ரவரி', 'மார்ச்', 'ஏப்ரல்']),
    'Bengali': dict(script='Bengali', title='মাসিক বিক্রয় প্রতিবেদন',
                    xlabel='মাস', ylabel='মূল্য',
                    months=['জানুয়ারি', 'ফেব্রুয়ারি', 'মার্চ', 'এপ্রিল']),
    'Chinese': dict(script='Han', title='月度销售报告',
                    xlabel='月份', ylabel='数值',
                    months=['一月', '二月', '三月', '四月']),
}

x = [1, 2, 3, 4]
y = [3, 5, 4, 6]
for language, g in GALLERY.items():
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.plot(x, y, marker='o')
    ur.set_multilingual_title(ax, g['title'], font_size=24)
    ur.set_multilingual_xlabel(ax, g['xlabel'], font_size=18)
    ur.set_multilingual_ylabel(ax, g['ylabel'], font_size=18)
    ur.set_multilingual_xticks(ax, x, g['months'], font_size=14)
    ur.set_multilingual_numeric_ticks(ax, script=g['script'], axis='y',
                                      positions=[3, 4, 5, 6], font_size=13)
    ur.apply_multilingual_layout(fig, auto=True)
    print(f'--- {language} ---')
    plt.show()

## 8. Try your own language ✏️

Edit `YOUR_TEXT` below — any language, any script, mixed is fine — and re-run the cell.

In [ ]:
YOUR_TEXT = 'नमस्ते Hello สวัสดี مرحبا নমস্কার 2026'   # <-- edit me!

info = ur.describe_text(YOUR_TEXT)
print('dominant script :', info['dominant_script'])
print('direction       :', info['direction'])
print('mixed scripts   :', info['is_mixed'])
print('chosen font     :', ur.auto_font_fallback(YOUR_TEXT))

fig, ax = plt.subplots(figsize=(8, 1.8))
ax.axis('off')
ur.multilingual_text(ax, 0.5, 0.5, YOUR_TEXT, coord='axes', font_size=40)
plt.show()

## 9. (Optional) How do the alternatives do on this machine?

`mplcairo` is the strongest existing alternative — it *can* shape complex text, but only when built with the Raqm library. On Windows its official wheels ship **without** Raqm and fail silently; this cell reports what happens on the current platform.

In [ ]:
%pip -q install mplcairo
import io
import matplotlib.image as mpimg
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties, findfont

try:
    import mplcairo
    from mplcairo.base import FigureCanvasCairo
    versions = mplcairo.get_versions()
    print('mplcairo versions:', versions)
    print('Raqm (shaping) available:', bool(versions.get('raqm')))

    def render_with_mplcairo(text, family):
        f = Figure(figsize=(6, 1.4), dpi=100)
        canvas = FigureCanvasCairo(f)
        try:
            fp = FontProperties(fname=findfont(FontProperties(family=family),
                                fallback_to_default=False), size=34)
        except Exception:
            fp = FontProperties(size=34)
        f.text(0.5, 0.5, text, ha='center', va='center', fontproperties=fp)
        buf = io.BytesIO()
        canvas.print_png(buf)
        buf.seek(0)
        return mpimg.imread(buf)

    tests = {'Hindi': 'विश्वविद्यालय', 'Arabic': 'جامعة القاهرة',
             'Bengali': 'বিশ্ববিদ্যালয়', 'Thai': 'มหาวิทยาลัย'}
    fig, axes = plt.subplots(len(tests), 2, figsize=(11, 2.0 * len(tests)))
    for i, (script, text) in enumerate(tests.items()):
        family = ur.auto_font_fallback(text)
        axes[i, 0].imshow(render_with_mplcairo(text, family))
        axes[i, 0].set_title(f'{script} — mplcairo', fontsize=11)
        arr = ur.render_text_array(text, font_size=40)
        axes[i, 1].imshow(arr)
        axes[i, 1].set_title(f'{script} — universal_render', fontsize=11)
        for a in axes[i]:
            a.axis('off')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print('mplcairo unavailable on this platform:', e)

## 10. Your feedback — 2 minutes of your time 🙏

If you read **any** of the languages shown above — Hindi, Arabic, Bengali, Tamil, Thai, Telugu, Hebrew, Korean, Chinese, or others from the self-test — your judgment is exactly what we need:

1. Look at the **before/after figure (section 4)** and the **gallery plot in your language (section 7)**.
2. Is the universal-render output correct and readable? Would you use it in a publication?

**Ways to tell us:**
- 📝 Short survey (preferred — takes ~2 min): **[SURVEY LINK — coming soon]**
- 📧 Email: **mrinalbasak23@gmail.com** (a screenshot + which language you read is enough)
- 🐛 GitHub issues (bugs, wrong rendering, or a language you want added): https://github.com/mbs57/universal-render/issues
- 💬 Or reply in the comments of the social-media post that brought you here

*universal-render generalizes bangla-render (SoftwareX) into a universal complex-script rendering framework. MIT licensed.*